[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/polars-certified/notebooks/day-02-expressions-and-the-polars-api.ipynb#scrollTo=10a2b3c4)

---
# Day 2 · Expressions and the Polars API
**certified-journeys / polars-certified** &nbsp;|&nbsp; Core API

> **Goal for today:** Master Polars expressions as the core building block — `pl.col()`, `pl.lit()`, `pl.when()`, and chained method calls.

---
## What is a Polars expression?

In Pandas, you manipulate data by referencing Series objects (`df["col"]`). In Polars, you write
**expressions** — lazy descriptions of what you want to compute — and Polars figures out *how*.

| Building block | What it does | Pandas equivalent |
|---|---|---|
| `pl.col("x")` | Reference a column by name | `df["x"]` |
| `pl.col(pl.Float64)` | Reference all Float64 columns | `df.select_dtypes(float)` |
| `pl.all()` | Reference every column | `df` |
| `pl.lit(42)` | A scalar constant as an expression | `42` (inline) |
| `pl.when().then().otherwise()` | Conditional expression | `np.where()` |

> **Key insight:** In Polars, expressions are the core primitive. Everything is built with `pl.col()` and chained methods. Internalize this and the whole API makes sense.

In [ ]:
%pip install -q polars numpy pandas

---
## Step 1 · `pl.col()` basics

`pl.col()` is the gateway to every column-level operation. It can target one column, multiple columns,
all columns of a dtype, or every column — all with the same syntax.

In [ ]:
import polars as pl
import numpy as np

df = pl.DataFrame({
    "name":     ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "dept":     ["eng",   "mkt", "eng",   "fin",  "mkt"],
    "salary":   [95_000,  72_000, 88_000, 110_000, 67_000],
    "bonus":    [0.10,    0.08,   0.12,   0.15,    0.07],
    "years":    [3,       1,      5,      8,        2],
})

# Single column — returns a 1-column DataFrame (not a Series) when used in select()
print("Single column:")
print(df.select(pl.col("name")))
print()

# Multiple columns by name
print("Multiple columns:")
print(df.select(pl.col("name", "salary")))
print()

# All columns of a specific dtype
print("All Float64 columns:")
print(df.select(pl.col(pl.Float64)))
print()

# All columns — pl.all() is shorthand for pl.col("*")
print("All columns (first 2 rows):")
print(df.select(pl.all()).head(2))

**What just happened?**
- `pl.col("name", "salary")` selects multiple columns in one expression — no list needed
- `pl.col(pl.Float64)` selects **by dtype** — powerful for applying operations to all numeric columns at once
- `pl.all()` is equivalent to `pl.col("*")` — selects every column
- `select()` always returns a **DataFrame**, even for one column — there's no implicit Series coercion
- **Every expression is composable** — you can chain `.alias()`, `.cast()`, math ops, etc. onto any `pl.col()`

---
## Step 2 · `select()` vs `with_columns()`

These two methods look similar but behave very differently:

| Method | Columns in output | Use when |
|---|---|---|
| `select()` | **Only** the columns you list | You want to project / reduce columns |
| `with_columns()` | **All** existing columns + new/updated ones | You want to add or update columns |

In [ ]:
# select() — output has ONLY the columns you specify
projected = df.select(
    pl.col("name"),
    pl.col("salary"),
    (pl.col("salary") * pl.col("bonus")).alias("bonus_amount"),  # derived column
)
print("select() — only listed columns:")
print(projected)
print()

# with_columns() — output keeps ALL columns, adds / replaces named columns
enriched = df.with_columns(
    (pl.col("salary") * pl.col("bonus")).alias("bonus_amount"),  # NEW column added
    (pl.col("salary") * (1 + pl.col("bonus"))).alias("total_comp"),  # another new column
)
print("with_columns() — all original columns + new ones:")
print(enriched)

**What just happened?**
- `select()` **projected** to 3 columns — the `dept`, `years`, `bonus` columns are gone from the result
- `with_columns()` **added** two new columns while keeping all original five
- Both methods accept the same expression syntax — the only difference is what's kept in output
- **Derived columns use arithmetic directly on `pl.col()` expressions** — no need to extract Series first
- You can pass **multiple expressions** in a single call — they all execute in parallel

---
## Step 3 · `filter()` with compound conditions

Polars uses Python bitwise operators (`&`, `|`, `~`) for combining filter conditions.
This is the same as Pandas, but the operands are Polars expressions, not boolean Series.

In [ ]:
# Single condition
senior = df.filter(pl.col("years") >= 3)
print("years >= 3:")
print(senior)
print()

# AND condition — both must be True
senior_eng = df.filter(
    (pl.col("years") >= 3) & (pl.col("dept") == "eng")
)
print("years >= 3 AND dept == eng:")
print(senior_eng)
print()

# OR condition — either must be True
eng_or_fin = df.filter(
    (pl.col("dept") == "eng") | (pl.col("dept") == "fin")
)
print("dept == eng OR fin:")
print(eng_or_fin)
print()

# NOT condition — invert
not_mkt = df.filter(~(pl.col("dept") == "mkt"))
print("NOT mkt (shorthand: is_in exclusion):")
print(not_mkt)
print()

# is_in — cleaner way to filter by a list of values
tech_finance = df.filter(pl.col("dept").is_in(["eng", "fin"]))
print("dept in [eng, fin] using is_in():")
print(tech_finance)

**What just happened?**
- Compound conditions **must wrap each sub-condition in parentheses** — operator precedence in Python requires this
- `~` negates a boolean expression — equivalent to Pandas `~` but on expressions
- **`is_in()`** is cleaner than multiple `|` conditions when checking against a list of values
- `filter()` accepts a **single expression** that evaluates to a Boolean column
- All conditions execute in a single pass over the data — no intermediate allocations

---
## Step 4 · `pl.when().then().otherwise()` and `pl.lit()`

`pl.when()` is Polars' replacement for `np.where()` and Pandas `.apply(lambda x: ...)` patterns.
It's vectorized and can be chained for multiple conditions (like SQL `CASE WHEN`).

`pl.lit()` wraps a scalar into an expression so it can be used alongside `pl.col()` in expressions.

In [ ]:
# pl.lit() — scalar as expression
with_tax_rate = df.with_columns(
    pl.lit(0.22).alias("tax_rate")   # add a constant column — pl.lit() wraps the scalar
)
print("pl.lit() adds a constant column:")
print(with_tax_rate.head(2))
print()

# pl.when().then().otherwise() — single condition (like np.where)
df_graded = df.with_columns(
    pl.when(pl.col("salary") >= 90_000)
      .then(pl.lit("senior"))
      .otherwise(pl.lit("standard"))
      .alias("pay_band")
)
print("Binary pay_band with pl.when():")
print(df_graded)
print()

# Chained pl.when() — multiple conditions (like SQL CASE WHEN ... WHEN ... ELSE)
df_tier = df.with_columns(
    pl.when(pl.col("years") >= 6)
      .then(pl.lit("principal"))     # condition 1
      .when(pl.col("years") >= 3)
      .then(pl.lit("senior"))        # condition 2
      .otherwise(pl.lit("junior"))   # default / else
      .alias("seniority")
)
print("Three-tier seniority with chained pl.when():")
print(df_tier)

**What just happened?**
- `pl.lit()` wraps a Python scalar into a Polars expression — required when mixing scalars with `pl.col()` in `.then()` / `.otherwise()`
- `pl.when().then().otherwise()` is **fully vectorized** — no Python loop, no `.apply()`
- **Chaining `.when().when()`** adds more conditions — conditions are evaluated top-to-bottom, first match wins
- The final `.otherwise()` is the `ELSE` clause — always include it to avoid unexpected nulls
- This pattern replaces `np.where()`, Pandas `.apply(lambda ...)`, and `pd.cut()` in most cases

---
## Step 5 · 5 Pandas operations → Polars

Side-by-side translation of the most common Pandas patterns:

In [ ]:
import pandas as pd

# Shared data
df_pd = df.to_pandas()   # convert Polars → Pandas for comparison

print("=== 1. Add a derived column ===")
# Pandas: df.assign()
pd_result = df_pd.assign(total_comp=df_pd["salary"] * (1 + df_pd["bonus"]))
# Polars: with_columns()
pl_result = df.with_columns(
    (pl.col("salary") * (1 + pl.col("bonus"))).alias("total_comp")
)
print(pl_result.head(2))

print("\n=== 2. Conditional column (np.where / pl.when) ===")
# Pandas: np.where()
pd_result = df_pd.assign(flag=np.where(df_pd["salary"] > 80_000, "high", "low"))
# Polars: pl.when().then().otherwise()
pl_result = df.with_columns(
    pl.when(pl.col("salary") > 80_000).then(pl.lit("high")).otherwise(pl.lit("low")).alias("flag")
)
print(pl_result.head(2))

print("\n=== 3. Row filter by label (loc) ===")
# Pandas: df.loc[df["dept"] == "eng"]
pd_result = df_pd.loc[df_pd["dept"] == "eng"]
# Polars: filter()
pl_result = df.filter(pl.col("dept") == "eng")
print(pl_result)

print("\n=== 4. Apply a function element-wise ===")
# Pandas: df["salary"].apply(lambda x: x * 1.1)
pd_result = df_pd["salary"].apply(lambda x: x * 1.1)
# Polars: expression — NO apply(), just arithmetic on pl.col()
pl_result = df.select((pl.col("salary") * 1.1).alias("salary_raised"))
print(pl_result.head(2))

print("\n=== 5. Rename a column ===")
# Pandas: df.rename(columns={"bonus": "bonus_rate"})
pd_result = df_pd.rename(columns={"bonus": "bonus_rate"})
# Polars: rename() OR alias() inside select()
pl_result = df.rename({"bonus": "bonus_rate"})
print(pl_result.head(2))

**What just happened?**
- **Never use `.apply()` in Polars** — express the logic as a vectorized expression instead, it's 50–100x faster
- `with_columns()` replaces Pandas `.assign()` — same concept, expression-based syntax
- `filter()` replaces `.loc[]` — no row index means filtering is always by condition, never by label
- `.alias()` inside `select()` is a clean way to rename during projection
- **The key shift:** in Polars you describe *what* you want; the engine decides *how* to compute it

---
## Step 6 · Chaining: one pipeline, one pass

Polars encourages long, readable chains. Each method returns a new DataFrame — chain them all.

In [ ]:
result = (
    df
    .filter(pl.col("salary") > 70_000)              # keep high earners
    .with_columns(
        (pl.col("salary") * pl.col("bonus")).alias("bonus_amount"),   # add derived column
        pl.when(pl.col("years") >= 4)                                   # add seniority tier
          .then(pl.lit("senior"))
          .otherwise(pl.lit("junior"))
          .alias("tier"),
    )
    .select(pl.col("name", "dept", "salary", "bonus_amount", "tier"))   # keep only needed cols
    .sort("salary", descending=True)                # sort highest salary first
)

print("Full pipeline result:")
print(result)

**What just happened?**
- The entire transformation is **one chained expression** — readable top-to-bottom like a SQL query
- Operations are executed in order: filter first (reduces rows), then compute new columns, then project, then sort
- **Polars evaluates each step eagerly** in this mode — in Day 3 you'll see lazy mode defer all of this and optimize
- Long chains are idiomatic Polars — wrap them in `()` to allow multi-line without backslash continuation

---
## Challenge — do this yourself

Using the DataFrame below, create a column `"category"` using **only** `pl.when().then().otherwise()` with 3 conditions:
- `total_comp > 120_000` → `"top_earner"`
- `total_comp > 90_000` → `"mid_earner"`
- otherwise → `"standard"`

Where `total_comp = salary * (1 + bonus)`. Build it in a single pipeline.

In [ ]:
# Your solution here
employees = pl.DataFrame({
    "name":   ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank"],
    "salary": [95_000, 72_000, 88_000, 110_000, 67_000, 130_000],
    "bonus":  [0.10,   0.08,   0.12,   0.15,    0.07,   0.20],
})

result = (
    employees
    # TODO: add total_comp column using with_columns
    # TODO: add category column using pl.when().then().otherwise() with 3 branches
)
print(result)

---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| `pl.col()` | Reference columns by name, list, dtype, or `"*"` for all |
| `pl.lit()` | Wrap a scalar into an expression for use in `.then()` / `.otherwise()` |
| `select()` | Projects — output has only the columns you list |
| `with_columns()` | Enriches — output keeps all columns, adds/updates named columns |
| `filter()` | Row filter by Boolean expression — compound with `&`, `|`, `~` |
| `pl.when().then().otherwise()` | Vectorized conditional — replaces `np.where` and `.apply(lambda)` |
| Chaining | All methods return new DataFrames — chain freely for readable pipelines |

> **Tip:** In Polars, expressions are the core primitive. Everything is built with `pl.col()` and chained methods. Avoid `.apply()` — always look for the vectorized expression equivalent.

---
## What's next

**Day 3** → Lazy evaluation and query optimization: `LazyFrame`, `scan_csv()`, `.explain()`, predicate pushdown, and timing comparisons.

Mark Day 2 complete in your [tracker](../index.html).